# YbF


This is aiming to reproduce and understand the results from [An ultracold molecular beam for testing fundamental physics](https://iopscience.iop.org/article/10.1088/2058-9565/ac107e)

## Loading packages

In [27]:
using Pkg
Pkg.activate("/Users/jose/Documents/Works/MIT/RaX/Simu/Molecule-Sims/")
using
    Revise,
    QuantumStates,            # for calculating molecular structure
    OpticalBlochEquations,    # for solving optical Bloch equations
    UnitsToValue,              # for numerical values
    DifferentialEquations, # for differential equations (with
    Plots,                    # for plotting
    PlotlyJS,
    LinearAlgebra,
    StaticArrays
;

  Activating project at `~/Documents/Works/MIT/RaX/Simu/Molecule-Sims`


# Defining the energy structure

### Ground state Hamiltonian for the $X^2\Sigma^+(N=1)$

Includes defining a magnetic field given by $$\vec B = B_0 (0, 1/\sqrt{2}, 1/\sqrt{2})$$

In [11]:
# Quantum Number Bounds
# Define the quantum number bounds for the system. Here, S, I, Λ, and N represent
# the spin, nuclear spin, projection of the electronic orbital angular momentum
# along the molecular axis, and the rotational quantum number, respectively.
QN_bounds = (
    S = 1/2,           # Spin quantum number
    I = 1/2,           # Nuclear spin quantum number
    Λ = 0,             # Projection of the electronic orbital angular momentum
    N = 0:3            # Rotational quantum number range
)

# Generate all possible quantum states for a Hund's case (b) linear molecule
X_state_basis = enumerate_states(HundsCaseB_LinearMolecule, QN_bounds)

# Define the Hamiltonian Operator
X_state_operator = :(
    BX * Rotation +                     # Rotational energy term
    DX * RotationDistortion +           # Rotational distortion energy term
    γX * SpinRotation +                 # Spin-rotation interaction term
    bFX * Hyperfine_IS +                # Hyperfine interaction (Fermi contact term)
    cX * (Hyperfine_Dipolar / 3)        # Hyperfine interaction (dipolar term)
)

# Define the Parameters for the Hamiltonian
# These parameters are specific to the YbF molecule and are given in Hertz (Hz).
X_state_parameters = QuantumStates.@params begin
    BX = 7233.8271e6         # Rotational constant in Hz
    DX = 0.                  # Rotational distortion constant in Hz (set to 0 for simplicity)
    γX = -13.41679e6         # Spin-rotation interaction constant in Hz
    bFX = 170.26374e6        # Fermi contact term in Hz
    cX = 85.4028e6           # Dipolar hyperfine interaction constant in Hz
end

# Create the Hamiltonian
# Combine the basis states, operator, and parameters to define the Hamiltonian.
X_state_ham = Hamiltonian(basis=X_state_basis, operator=X_state_operator, parameters=X_state_parameters)

# Define Zeeman Terms
# The Zeeman effect describes how energy levels are split in the presence of a magnetic field.
# These functions define the interaction with the magnetic field along x, y, and z directions.
Zeeman_x(state, state′) = (1/√2) * (Zeeman(state, state′, -1) - Zeeman(state, state′, +1))
Zeeman_y(state, state′) = (im/√2) * (Zeeman(state, state′, -1) + Zeeman(state, state′, +1))
Zeeman_z(state, state′) = Zeeman(state, state′, 0)

# Set Magnetic Field Components
# Set the components of the magnetic field (in Tesla, converted to Gauss).
B = 1.0
X_state_ham.parameters.B_x = B * 1e-4  # Convert Tesla to Gauss
X_state_ham.parameters.B_y = B * 1e-4  # Convert Tesla to Gauss

# Add Zeeman Interaction Terms to the Hamiltonian
# Include the interaction terms for the magnetic field in the Hamiltonian.
X_state_ham = add_to_H(X_state_ham, :B_x, gS * μB * Zeeman_x)
X_state_ham = add_to_H(X_state_ham, :B_y, gS * μB * Zeeman_y)
X_state_ham = add_to_H(X_state_ham, :B_z, gS * μB * Zeeman_z)

# Evaluate and Solve the Hamiltonian
# Compute the energy levels and eigenstates of the Hamiltonian.
evaluate!(X_state_ham)
QuantumStates.solve!(X_state_ham)
all_grounds = X_state_ham.states;


### Create Hamiltonian for the $A^2\Pi_{1/2}(000, J=1/2+)$ state

In [13]:
# Define quantum number bounds for the excited state (A state)
QN_bounds = (
    S = 1/2,            # Electron spin quantum number
    I = 1/2,            # Nuclear spin quantum number
    Λ = (-1,1),         # Projection of electronic orbital angular momentum (with possible values -1 and 1)
    J = 1/2:3/2         # Total angular momentum quantum number range
)

# Enumerate states for the excited state basis
A_state_basis = enumerate_states(HundsCaseA_LinearMolecule, QN_bounds)

# Define the Hamiltonian operator for the excited state
A_state_operator = :(
    T_A * DiagonalOperator +     # Diagonal constant (electronic zero point energy)
    Be_A * Rotation +            # Rotational term
    Aso_A * SpinOrbit +          # Spin-orbit interaction term
    q_A * ΛDoubling_q +          # Λ-doubling term q
    p_A * ΛDoubling_p2q +        # Λ-doubling term p2q
    a * Hyperfine_IL +           # Hyperfine interaction (IL term)
    d * Hyperfine_Dipolar_d      # Hyperfine interaction (dipolar term)
)

# Define parameters for the excited state Hamiltonian
A_state_parameters = QuantumStates.@params begin
    T_A = 542.8102*10^12      # Diagonal constant (electronic zero point energy) in Hz
    Be_A = 40.9304*10^9       # Rotational constant in Hz
    Aso_A = 7.4276*10^12      # Spin-orbit interaction constant in Hz
    p_A = 11.882*10^9         # Λ-doubling constant p in Hz
    q_A = 0                   # Λ-doubling constant q (set to 0 for simplicity)
    a = -0.8e6                # Hyperfine IL interaction constant in Hz
    d = -4.6e6                # Dipolar hyperfine interaction constant in Hz
end

# Create the excited state Hamiltonian
A_state_ham = Hamiltonian(basis=A_state_basis, operator=A_state_operator, parameters=A_state_parameters)

# Evaluate and solve the Hamiltonian for the excited state
evaluate!(A_state_ham)
QuantumStates.solve!(A_state_ham)

In [19]:
# Convert relevant A states (J=1/2+) to a Hund's case (b) basis

# We used a Hund's case (a) basis to define the Hamiltonian for the excited A state.
# In order to compute transition dipole moments with the ground state, we wish to convert these states to a Hund's case (b).
# This is accomplished by defining a new basis that includes all the Hund's case (b) states required for this conversion,
# and then using the function `convert_basis`.

# Select the relevant A state (J=1/2+) states from the Hamiltonian solutions
A_state_J12_pos_parity_states = A_state_ham.states[5:8]

# Define quantum number bounds for the A state in Hund's case (b)
QN_bounds = (
    S = 1/2,            # Electron spin quantum number
    I = 1/2,            # Nuclear spin quantum number
    Λ = (-1,1),         # Projection of electronic orbital angular momentum (with possible values -1 and 1)
    N = 0:3             # Rotational quantum number range
)

# Enumerate states for the A state basis in Hund's case (b)
A_state_caseB_basis = enumerate_states(HundsCaseB_LinearMolecule, QN_bounds)

# Select the ground state solutions from the Hamiltonian
# Note: This is dropping all the terms with N = 0. (We only want N = 1)
ground_states = X_state_ham.states[5:16]

# Convert the excited states to Hund's case (b) basis
excited_states = convert_basis(A_state_J12_pos_parity_states, A_state_caseB_basis)

# Combine ground and excited states
states = [ground_states; excited_states];


In [22]:
Aenergies = [energy(st) for st in excited_states]
Xenergies = [energy(st) for st in ground_states]
A_Js = [st.basis[i].J for (i, st) in enumerate(excited_states)]
X_Js = [st.basis[i].J for (i, st) in enumerate(ground_states)]
A_Ns = [st.basis[i].N for (i, st) in enumerate(excited_states)]
X_Ns = [st.basis[i].N for (i, st) in enumerate(ground_states)]
A_Fs = [st.basis[i].F for (i, st) in enumerate(excited_states)]
X_Fs = [st.basis[i].F for (i, st) in enumerate(ground_states)]
df = DataFrame(
    J = [A_Js; X_Js],
    N = [A_Ns; X_Ns],
    F = [A_Fs; X_Fs],
    energy = [Aenergies; Xenergies],
    state = [["A" for _ in Aenergies]; ["X" for _ in Xenergies]]
);


In [25]:
# Initialize matrices to store dipole moments
d = zeros(ComplexF64, 16, 16, 3)    # Matrix to store all dipole moments (16x16 matrix for 3 components)
d_ge = zeros(ComplexF64, 12, 4, 3)  # Matrix to store dipole moments between ground and excited states (12x4 matrix for 3 components)

# Compute transition dipole moments between two bases
# basis_tdms stores the transition dipole moments between the ground and excited state bases
basis_tdms = get_tdms_two_bases(X_state_ham.basis, A_state_caseB_basis, TDM)

# Calculate transition dipole moments between specific states
# This function fills the d_ge matrix with the dipole moments between the ground_states and excited_states
tdms_between_states!(d_ge, basis_tdms, ground_states, excited_states)

# Assign the calculated dipole moments to the appropriate submatrix in d
# The ground to excited state transitions are stored in the upper right submatrix of d
d[1:12, 13:16, :] .= d_ge;


# Structure searching

In [29]:
# We use the functions to acces the states of the systems and search by the quantum numbers

In [33]:
# Put energy data in a DataFrame and plot it using `PlotlyJS`.
using PlotlyJS
Aenergies = [energy(st) for st in excited_states]
Xenergies = [energy(st) for st in all_grounds]

ground_quantum_numbers = [extract_quantum_numbers_from_state(st) for st in all_grounds]
excited_quantum_numbers = [extract_quantum_numbers_from_state(st) for st in excited_states]

# For all_grounds
X_Js = [qn[:J] for qn in ground_quantum_numbers]
X_Ms = [qn[:M] for qn in ground_quantum_numbers]
X_Fs = [qn[:F] for qn in ground_quantum_numbers]
X_Ns = [qn[:N] for qn in ground_quantum_numbers]

# For excited_states
A_Js = [qn[:J] for qn in excited_quantum_numbers]
A_Ms = [qn[:M] for qn in excited_quantum_numbers]
A_Fs = [qn[:F] for qn in excited_quantum_numbers]
A_Ns = [qn[:N] for qn in excited_quantum_numbers]

# Create a DataFrame
using DataFrames

df = DataFrame(
    J = [A_Js; X_Js],
    M = [A_Ms; X_Ms],
    F = [A_Fs; X_Fs],
    N = [A_Ns; X_Ns],
    energy = [Aenergies; Xenergies],
    state = [["A" for _ in Aenergies]; ["X" for _ in Xenergies]]
)

# Sort the DataFrame by energy and then M
sort!(df, [:energy, :M, :F, :J, :N]);

In [15]:
# # Add a different marker for different M values and label each point by the quantum numbers
# PlotlyJS.plot(
#     PlotlyJS.scatter(
#         x = df.N[df.state .== "X"],
#         y = df.energy[df.state .== "X"],
#         mode = "markers",
#         marker = attr(
#             size = 10,
#             color = df.M,
#             colorscale = "Viridis",
#             colorbar = attr(title = "M"),
#             showscale = true
#         ),
#         text = ["J: $(df.J[i]), M: $(df.M[i]), F: $(df.F[i]), N: $(df.N[i])" for i in 1:length(df.energy)],
#         hoverinfo = "text",
#         name = "A and X states",
#         type = "scatter"
#     ),
# )

### Compute dipole moments between states

In [16]:
d = zeros(ComplexF64, 16, 16, 3)
d_ge = zeros(ComplexF64, 12, 4, 3)

basis_tdms = get_tdms_two_bases(X_state_ham.basis, A_state_caseB_basis, TDM)
tdms_between_states!(d_ge, basis_tdms, ground_states, excited_states)
d[1:12, 13:16, :] .= d_ge
;

In [17]:
# A few constants used for the simulation
λ = 552e-9 # Wavelength of light in meters
Γ = 2π * 5.7e6
m = @with_unit 191 "u" # Mass of the molecule in atomic mass units
k = 2π / λ
;

# Setup the laser scheme

source states:

 - All N=1 states of the $X^2\Sigma^+(000)$ state

Target states:

 - State $J=1/2+$ states of the $A^2\Pi_{1/2}(000)$ with $N=1$

In [18]:
# source states
input_qn = [elem for elem in ground_quantum_numbers if elem[:N] == 1]# && elem[:J] == 1/2]#[1]
input_states = [search_state_by_quantum_numbers(all_grounds, elem[:N], elem[:J], elem[:F], elem[:M]) for elem in input_qn]
input_energies = [energy(st) for st in input_states]

# Target states
a_qns =[extract_quantum_numbers(st) for st in excited_states]
output_qn = [elem for elem in excited_quantum_numbers if elem[:J] == 1/2]#[1]
target_energy = [energy(st) for st in excited_states if extract_quantum_numbers(st) == output_qn[1]];

In [19]:
delta_state_energies = target_energy .- input_energies

12-element Vector{Float64}:
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14
 5.391432712666651e14

We need to define the lasers that we wish to consider for our simulation. This is accomplished using the type `Field`, which takes a (unit!) $k$ vector, a function to set the polarization (defined in the spherical basis), a laser frequency (in angular units), and a saturation parameter. 

In [20]:
# using StaticArrays
# using LinearAlgebra

# # Define linear polarizations along the x and y axes
# const lin_x = SVector{3, ComplexF64}(1.0, 0.0, 0.0)
# const lin_y = SVector{3, ComplexF64}(0.0, 1.0, 0.0)

# # Rotation function for linear polarization
# function linear_rotate_pol(pol::SVector{3, Float64}, k::SVector{3, Float64})::SVector{3, ComplexF64}
#     k /= norm(k)
#     x, y, z = cross(k, ẑ)
#     θ = acos(k ⋅ ẑ)
#     y = -y
#     if θ ≈ 0
#         α = 0.0
#         β = 0.0
#         γ = 0.0
#     elseif θ ≈ π
#         α = 0.0
#         β = π
#         γ = 0.0
#     else
#         A33 = (1 - cos(θ)) * z^2 + cos(θ)
#         A31 = (1 - cos(θ)) * z * x - y * sin(θ)
#         A32 = (1 - cos(θ)) * z * y + x * sin(θ)
#         A13 = (1 - cos(θ)) * x * z + y * sin(θ)
#         A23 = (1 - cos(θ)) * y * z - x * sin(θ)

#         α = atan(A23, A13)
#         β = atan(sqrt(1 - A33^2), A33)
#         γ = atan(A32, -A31)
#     end
#     return inv(D(β, α, γ)) * pol
# end

# # Define D function for rotation matrix
# function D(β, α, γ)
#     m = [[cos(α)*cos(γ) - sin(α)*cos(β)*sin(γ), -cos(α)*sin(γ) - sin(α)*cos(β)*cos(γ), sin(α)*sin(β)],
#          [sin(α)*cos(γ) + cos(α)*cos(β)*sin(γ), -sin(α)*sin(γ) + cos(α)*cos(β)*cos(γ), -cos(α)*sin(β)],
#          [sin(β)*sin(γ), sin(β)*cos(γ), cos(β)]]
#     # CONVERT TO A matrix
#     return mapreduce(permutedims, vcat, m)

# end

# # Initialize polarizations with linear polarizations
# polarizations = [(1/(sqrt(2))) * (lin_x + lin_y) for _ in input_states]
# # convert to reals
# polarizations = [SVector{3, Float64}(pol) for pol in polarizations]


12-element Vector{SVector{3, Float64}}:
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]
 [0.7071067811865475, 0.7071067811865475, 0.0]

In [21]:
s_func(s) = (r,t) -> s
s = s_func(10.0)

A_energy = target_energy[1]
period = 1/Γ
omega = 2π / period

detunning = 6

# XY ll
δs = [+detunning*Γ for _ in input_states]
ωs = [2π * (A_energy - input_energies[i]) + δs[i] for i in 1:length(input_states)]
ϵ(ϵ_val) = t -> ϵ_val*cos(omega*t)


ϵ (generic function with 1 method)

In [22]:

# x axis
k̂ = +x̂; ϵs = [linear_rotate_pol(polarizations[j], k̂) for j in 1:length(polarizations)];
lasers_x = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

# y axis
k̂ = +ŷ; ϵs = ϵs = [linear_rotate_pol(polarizations[j], k̂) for j in 1:length(polarizations)];
lasers_y = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

lasers = [lasers_x; lasers_y]
;

In [23]:
# const σ⁻ = SVector{3, ComplexF64}(1.0, 0.0, 0.0)
# const σ⁺ = SVector{3, ComplexF64}(0.0, 0.0, 1.0)
# const σ⁰ = SVector{3, ComplexF64}(0.0, 1.0, 0.0)

# function rotate_pol(pol, k)::SVector{3,Complex{Float64}}
#     k /= norm(k)
#     x, y, z = cross(k, ẑ)
#     θ = acos(k ⋅ ẑ)
#     y = -y
#     if θ ≈ 0
#         α = 0.0
#         β = 0.0
#         γ = 0.0
#     elseif θ ≈ π
#         α = 0.0
#         β = π
#         γ = 0.0
#     else

#         A33 = (1 - cos(θ)) * z^2 + cos(θ)
#         A31 = (1 - cos(θ)) * z * x - y * sin(θ)
#         A32 = (1 - cos(θ)) * z * y + x * sin(θ)
#         A13 = (1 - cos(θ)) * x * z + y * sin(θ)
#         A23 = (1 - cos(θ)) * y * z - x * sin(θ)

#         α = atan(A23, A13)
#         β = atan(sqrt(1 - A33^2), A33)
#         γ = atan(A32, -A31)

#         A12 = x * y * (1 - cos(θ)) - z * sin(θ)
#         A21 = y * x * (1 - cos(θ)) + z * sin(θ)
#         A22 = cos(θ) + y^2 * (1 - cos(θ))
#         A11 = cos(θ) + x^2 * (1 - cos(θ))
#     end
#     return inv(D(cos(β), sin(β), α, γ)) * pol
# end
# polarizations = [σ⁺ for _ in input_states]
# ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]

In [24]:

# # x axis
# k̂ = +x̂; ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]
# lasers_plus_x = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]
# k̂ = -x̂; ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]
# lasers_minus_x = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

# # y axis
# k̂ = +ŷ; ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]
# lasers_plus_y = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]
# k̂ = -ŷ; ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]
# lasers_minus_y = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

# lasers = [lasers_plus_x; lasers_minus_x; lasers_plus_y; lasers_minus_y]
# ;

The OBE solver requires a set of parameters `p`, which are defined using the function `obe`. Most of these arguments should be self-explanatory, but a few require special attention:
* `freq_res` _(type: Float64)_: The resolution used when rounding frequencies (and the velocity) in order to ensure that the OBEs reach a quasi-periodic steady state. Defined in units of Γ and typically set to 1e-1 (or 1e-2 for low velocities).
* `extra_p` _(type: NamedTuple)_: Allows the user to define a set of "extra" parameters that may be used during the simulation. This is mostly relevant in cases where we wish to solve the OBEs several times and we have some parameter that is changing between solves, e.g., we may need to update our TDMs if we want to solve the OBEs for a range of magnetic fields. In that case it's helpful to ensure that the solver has access to the underlying Hamiltonian, as well as any data structures we've defined to help in updating the TDMs.

In [25]:
# Set initial conditions
particle = OpticalBlochEquations.Particle()
ρ0 = zeros(ComplexF64, length(states), length(states))
names = [extract_quantum_numbers(st) for st in states]
string_names = ["|$(qn[:J]),  $(qn[:F]), $(qn[:M])>" for qn in names]
ρ0[1,1] = 1.0

freq_res = 1e-1
p = obe(ρ0, particle, states, lasers, d, true, true; λ=λ, Γ=Γ, freq_res=freq_res)
;

After defining `p`, we're free to update any of its parameters before solving the OBEs. For example, we can update the initial position `r0` to $\mathbf{r}_0 = (0,0,0.5)~k^{-1}$ and the velocity `v` to $\mathbf{v} = (0,0,1)~\text{m/s}$ as shown below. Note that position and velocity need to have units of 1/k and Γ/k in the simulation, so since we defined the velocity in SI units it must be divided by Γ/k. We also need to round the velocity again (using `round_vel`) after updating it (this rounding is automatically performed inside `obe` when `p` is first defined but must be performed again if `v` is changed).

In [26]:
p.r0 = (0., 0., 0.5)
p.v = (0., 0., 1.0) ./ (Γ / k)
p.v = round_vel(p.v, p.freq_res)
;

Finally, we define a time span for the solver (here in terms of the period defined by `freq_res`, which is given by `p.period`). The actual solving is performed using the `DifferentialEquations` package in Julia, so we load this here as well. This package requires a `prob` variable of type `ODEProblem`. The cell below shows the syntax for how this variable is created for the OBE solver.

After solving, we can plot the populations as a function of time, and the force averaged over one period can be found in `prob.p.force_last_period`. Note that the force output in the simulation has units ħkΓ.

In [27]:
using DifferentialEquations
using Plots
t_end = 50p.period+1
tspan = (0., t_end)
prob = ODEProblem(ρ!, p.ρ0_vec, tspan, p)
times = range(0, t_end, 10_000)

cb = PeriodicCallback(reset_force!, p.period)
@time sol = DifferentialEquations.solve(prob, DP5(), callback=cb, reltol=1e-3, saveat=times)

# Print the force
println()
print("Force (10³ m/s): ", prob.p.force_last_period * (1e-3 * ħ * k * Γ / m))

plot_us = sol.u
plot_ts = sol.t

n_states = size(p.ρ_soa, 1)
Plots.plot(
    ylim=(-0.1, 1.6),
)

for i in 1:n_states
    state_idx = n_states*(i-1) + i # For the diagonal elements
    Plots.plot!(plot_ts, [real(u[state_idx]) for u in plot_us], label=string_names[i])
end
# Add the correct axis labels
Plots.plot!(
    xlabel="Time (s)",
    ylabel="Population"
)
# Add the legends to each plot


MethodError: MethodError: objects of type SVector{3, ComplexF64} are not callable
Use square brackets [] for indexing an Array.

In [28]:
pop_df = []
for i in 1:n_states
    _df = DataFrame(
        time = plot_ts,
        population = [real(u[n_states*(i-1) + i]) for u in plot_us],
        state = string_names[i]
    )
    push!(pop_df, _df)
end
pop_df = vcat(pop_df...)

UndefVarError: UndefVarError: `n_states` not defined

In [29]:
# using PlotlyJS
# using Colors
# unique_states = unique(pop_df.state)
# color_map = Dict{String, RGB}()
# for (i, state) in enumerate(unique_states)
#     color_map[state] = RGB(convert(Color, HSL(360 * (i - 1) / length(unique_states), 1.0, 0.5)))
# end
# traces = AbstractTrace[]  # Initialize as an empty array of AbstractTrace
# for state in unique_states
#     push!(traces, PlotlyJS.scatter(
#         x = pop_df.time[pop_df.state .== state],
#         y = pop_df.population[pop_df.state .== state],
#         mode = "lines",
#         text = pop_df.state[pop_df.state .== state],
#         hoverinfo = "text",
#         name = "Population: $state",
#         line = attr(color=PlotlyBase.Colors.hex(color_map[state]))
#     ))
# end
# layout = Layout(title="Population over Time by State", xaxis_title="Time", yaxis_title="Population")
# plot = Plot(traces, layout)
# display(plot)


### Force versus velocity

Creating a "force profile" (i.e., the force as a function of some variable) can be performed by combining the previous section with the function `force_scan_v3`. The function takes in a `prob` for the OBEs (which we already defined above) and a `scan_values_grid` (essentially a grid of all the values we wish to scan over). Note that `scan_values_grid` may be multidimensional, e.g., we may wish to scan over both position and velocity simultaneously. Indeed, a profile of force versus velocity typically involves averaging over a collection of different starting positions from $\mathbf{r} = (0,0,0)$ to $\mathbf{r} = (2\pi, 2\pi, 2\pi)$ (in units of 1/k), or $\mathbf{r} = (0,0,0)$ to $\mathbf{r} = (\lambda, \lambda, \lambda)$ (in SI units). This ensures that the force is averaged over all possible initial values of the combined laser field seen by the molecule.

Two additional arguments are also required for `force_scan_v3`:
* `prob_func!` _(prob, scan_params, i) -> prob_: Updates the initial `prob` for the `i`th value of the `scan_values_grid`.
* `output_func` _(p, sol) -> force_: Define the output of the solver after `prob` has been solved, typically the force.

In [30]:
using
    StaticArrays,
    RectiGrids,
    StatsBase

In [31]:
function prob_func!(prob, scan_values_grid, i)
    p = prob.p
    # Update velocity and position
    p.v .= (0, 0, scan_values_grid[i].v)
    p.v .= round_vel(p.v, p.freq_res)
    p.r0 .= scan_values_grid[i].r
    return prob
end
function output_func(p, sol)
    f = p.force_last_period
    return (0, 0, f[3])
end
;

In [32]:
freq_res = 1e-1
p = obe(ρ0, particle, states, lasers, d, true, true; λ=λ, Γ=Γ, freq_res=freq_res)

t_end = 10p.period+1; tspan = (0., t_end)
prob = ODEProblem(ρ!, p.ρ0_vec, tspan, p, reltol=1e-3, save_on=false)

di = 2
rs = vcat([(n1/(di+1), n2/(di+1), n3/(di+1)) .* 2π for n1 ∈ 0:di, n2 ∈ 0:di, n3 ∈ 0:di]...)
vs = -7:0.5:7.0

scan_values = (r = rs, v = vs)
scan_values_grid = RectiGrids.grid(scan_values)
;

In [33]:
@time forces, populations = force_scan_v2(prob, scan_values_grid, prob_func!, output_func);

CompositeException: TaskFailedException

    nested task error: MethodError: objects of type SVector{3, ComplexF64} are not callable
    Use square brackets [] for indexing an Array.
    Stacktrace:
      [1] update_fields!(fields::StructArrays.StructVector{Field{Float64, SVector{3, ComplexF64}, var"#79#80"{Float64}}, NamedTuple{(:k, :ϵ, :ϵ_val, :ω, :s_func, :s, :re, :im, :kr, :E), Tuple{Vector{SVector{3, Float64}}, Vector{SVector{3, ComplexF64}}, Vector{SVector{3, ComplexF64}}, Vector{Float64}, Vector{var"#79#80"{Float64}}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Vector{Float64}, Vector{SVector{3, ComplexF64}}}}, Int64}, r::MVector{3, Float64}, t::Float64)
        @ OpticalBlochEquations ~/.julia/packages/OpticalBlochEquations/JbHm6/src/field.jl:51
      [2] update_H_obes!(p::MutableNamedTuples.MutableNamedTuple{(:H, :particle, :ρ0, :ρ0_vec, :ρ_soa, :dρ_soa, :Js, :eiωt, :ω, :states, :fields, :r0, :r, :v, :Γ, :tmp, :λ, :period, :k, :freq_res, :H₀, :force_last_period, :popula

In [34]:
averaged_forces = Float64[]
@time for (i,v) ∈ enumerate(vs)
    idxs = [j for (j,x) ∈ enumerate(scan_values_grid) if x.v == v]
    push!(averaged_forces, mean([f[3] for f in forces[idxs]]))
end

UndefVarError: UndefVarError: `forces` not defined

In [35]:
Plots.plot(vs, (1e-3 * ħ * k * Γ / m) .* averaged_forces,
    xlabel="Velocity (m/s)",
    ylabel="a (10³ m/s²)",
    title="Blue detuned",
    framestyle=:box,
    linewidth=2.5,
    legend=nothing
    )